# 结构预测和处理
- /home/admin123/work/GTmining/diffpool/predict_data/

## 结构预测

In [ ]:
# python /home/admin123/esmfold/esm-main/scripts/fold.py -i domain_out.fasta -o structure -m /home/admin123/.cache/torch/hub/ --num-recycles 4

## 结构对齐

In [1]:
# ==========脚本内容：Protein_align.py==========
import subprocess
import os
from tqdm import tqdm

def get_all_files(directory):
    all_files = []
    # 遍历目录及其子目录
    for root, dirs, files in os.walk(directory):
        # 将每个文件的绝对路径添加到列表中
        for file in files:
            file_path = os.path.join(root, file)
            all_files.append(file_path)
    return all_files

def delete_files_in_folder(folder_path):
    # 获取文件夹下所有文件和子文件夹
    for root, dirs, files in os.walk(folder_path):
        # 删除所有文件
        for file in files:
            file_path = os.path.join(root, file)
            os.remove(file_path)

def delete_special_files_in_folder(folder_path, prefix, fila_n):
    # 获取文件夹下所有文件和子文件夹
    for root, dirs, files in os.walk(folder_path):
        # 删除所有文件
        for file in files:
            if file.endswith(prefix) and file.startswith(fila_n):
                file_path = os.path.join(root, file)
                os.remove(file_path)

fold_type = 'GTB'

directory = f"./predict_data/structure/"
storage_directory = f"./predict_data/structure_align/"
os.makedirs(storage_directory, exist_ok=True)
all_files = get_all_files(directory)
delete_files_in_folder(storage_directory)
# USalign结构比对==========
exe_path = './predict_data/exe/USalign'
cluster_center_pdb = f'./predict_data/cazy_cluster_center_{fold_type}.pdb'
# USalign AAA.pdb cluster.pdb -o AAA
for file in tqdm(all_files):
    file = file.split('/')[-1]
    temp1 = os.path.join(directory, file)
    temp2 = os.path.join(storage_directory, file.split('.pdb')[0])
    subprocess.run([exe_path, temp1, cluster_center_pdb, '-o', temp2], stdout=subprocess.DEVNULL)
    delete_special_files_in_folder(storage_directory, '.pml', file.split('.pdb')[0])


  0%|          | 0/217 [00:00<?, ?it/s]

100%|██████████| 217/217 [00:41<00:00,  5.21it/s]


# 提取特征
- 切换为环境starG

## Step1 计算网格参数

In [2]:
# 提取局部结构特征第一步
import os

fold_type = 'GTB'

structure_path = './predict_data/structure_align/'
files = os.listdir(structure_path)

f = open('./predict_data/Protein_extract_feature_step1.sh', 'w')
for file in files:
    f.write(f"python ../../Predict_Extract_feature_step1.py {file} 6 6 {fold_type}\n")
f.close()


# nohup parallel --jobs 30 < Protein_extract_feature_step1.sh > Feature_Step_1.out 2>&1 &


## Step2 获取局部特征

In [3]:
# 提取局部结构特征第二步
import os

fold_type = 'GTB'

structure_path = './predict_data/structure_align/'
files = os.listdir(structure_path)

sample_redies_udp = 6
sample_redies_sugar = 6

step = 1
epoch = 1
f = open(f'./predict_data/Extract_feature_step2_epoch{epoch}.sh', 'w')
for file in files:
    if step % 40 == 0:
        f.write(f"taskset -c {step} python ../../Predict_Extract_feature_step2.py {file} {sample_redies_udp} {sample_redies_sugar} {fold_type}\n")
        f.close()
        epoch += 1
        f = open(f'./predict_data/Extract_feature_step2_epoch{epoch}.sh', 'w')
        step = 1
        continue
    f.write(f"taskset -c {step} python ../../Predict_Extract_feature_step2.py {file} {sample_redies_udp} {sample_redies_sugar} {fold_type}\n")
    step = step + 1
f.close()

f = open(f'./predict_data/Extract_feature_step2_AAArun.sh', 'w')
for e in range(epoch):
    f.write(f"parallel --jobs 40 < Extract_feature_step2_epoch{e+1}.sh\n")
f.close()

# nohup bash Extract_feature_step2_AAArun.sh &


# 生成数据集

In [7]:
'''
根据DiffPool的教程，我需要准备的东西有4样，分别是A、graph_indicator、graph_labels、node_attributes
'''
import numpy as np
import os
from tqdm import tqdm
import shutil
import pandas as pd

fold_type = 'GTB'

sample_redies_udp = 6
sample_redies_sugar = 6

if fold_type == 'GTA':
    graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                        'UDP-Gal': 3, 'UDP-GalNAc': 4,
                        'UDP-Xyl': 5, 'GDP-Man': 6,
                        'dTDP-Rha': 7, 'Other': 8}
elif fold_type == 'GTB':
    graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                        'UDP-Gal': 3, 'UDP-GalNAc': 4,
                        'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                        'dTDP-Rha': 8, 'Other': 9}
else:
    raise ValueError('Fold_type must be one of the GTA and GTB.')


# 获取有哪些local结构
storage_path = './predict_data/local_feature/'
dl_data_path = './predict_data/dl_data/'

local_list = [x.split('.npy')[0] for x in os.listdir(storage_path)]

# 生成文件列表
predict_process_path = []
for file in os.listdir(storage_path):
    npy_path = os.path.join(storage_path, file)
    predict_process_path.append(npy_path)

# 管理文件夹
if not os.path.isdir(dl_data_path):
    os.makedirs(dl_data_path, exist_ok=True)
else:
    shutil.rmtree(dl_data_path)
    os.makedirs(dl_data_path, exist_ok=True)
os.makedirs(f'{dl_data_path}/predict/')
os.makedirs(f'{dl_data_path}/predict/trace_file/')

# ==================== 生成数据 ====================
def make_database(process_path: list, data_type: str):
    f_A = open(f'{dl_data_path}/{data_type}/GTmining_A.txt', 'w')
    f_graph_indicator = open(f'{dl_data_path}/{data_type}/GTmining_graph_indicator.txt', 'w')
    f_graph_labels = open(f'{dl_data_path}/{data_type}/GTmining_graph_labels.txt', 'w')
    f_node_attributes = open(f'{dl_data_path}/{data_type}/GTmining_node_attributes.txt', 'w')
    f_itemID = open(f'{dl_data_path}/{data_type}/GTmining_itemID.txt', 'w')  # 新增：itemID文件句柄
    f_correspond_information = open(f'{dl_data_path}/{data_type}/Predict_correspond_information.txt', 'w')
    edge_max = -1
    graph_idx = -1
    # 已经重构了数据处理部分，读取一次npy文件，写入四种不同的文件，提升I/O性能
    for file in tqdm(process_path):
        # +++++ 一次性读取npy文件 +++++
        try:
            input_dict = np.load(file, allow_pickle=True)
        except:
            print(f"wrong {file}")
            continue
        input_dict = input_dict[()]
        if len(input_dict['edges']) <= 100:
            # 用来检查不正确的局部网格
            print(f"error local feature in {file.split('/')[-1].split('.npy')[0]}")
            continue
        # 获取当前文件的唯一标识作为item ID
        item_id = file.split('/')[-1].split('.npy')[0]
        # +++++ f_A的数据处理 +++++
        edges = input_dict['edges']
        edges += (edge_max +1)
        for edge in edges:
            f_A.write(f"{edge[0]}, {edge[1]}\n")
        edge_max = np.max(edges)
        # +++++ f_graph_indicator的数据处理 +++++
        graph_idx += 1
        xyzs = input_dict['xyz']
        for i in range(0, xyzs.shape[0]):
            f_graph_indicator.write(f"{graph_idx}\n")
        # +++++ f_graph_labels的数据处理 +++++
        f_graph_labels.write("0\n")
        # +++++ 新增：f_itemID的数据处理 +++++
        # 每个图对应一行item ID
        f_itemID.write(f"{item_id}\n")
        # +++++ f_node_attributes的数据处理 +++++
        xyzs = input_dict['xyz']
        sis = input_dict['si']
        hbonds = input_dict['hbond']
        charges = input_dict['charge']
        hphobs = input_dict['hphob']
        trace_temp = file.split('/')[-1] + f'==={edge_max}.txt'
        f_trace_file = open(f'{dl_data_path}/{data_type}/trace_file/{trace_temp}', 'w')
        for i in range(0, xyzs.shape[0]):
            # x = xyzs[i][0] + np.random.normal(loc=0, scale=0.75)
            # y = xyzs[i][1] + np.random.normal(loc=0, scale=0.75)
            # z = xyzs[i][2] + np.random.normal(loc=0, scale=0.75)
            x = xyzs[i][0]
            y = xyzs[i][1]
            z = xyzs[i][2]
            f_trace_file.write("{:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z))
            f_node_attributes.write("{:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}, {:>10.6f}\n".format(x, y, z, sis[i][0], hbonds[i][0], charges[i][0], hphobs[i][0]))
        f_trace_file.close()
        f_correspond_information.write(file.split('/')[-1].split('.npy')[0] + '===')
        temp_correspond_information = ''
        temp_correspond_information = temp_correspond_information + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[0][0], 5),
                                                                                                            round(xyzs[0][1], 5),
                                                                                                            round(xyzs[0][2], 5),
                                                                                                            round(sis[0][0], 5),
                                                                                                            round(hbonds[0][0], 5),
                                                                                                            round(charges[0][0], 5),
                                                                                                            round(hphobs[0][0], 5))
        temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(xyzs[1][0], 5),
                                                                                                                round(xyzs[1][1], 5),
                                                                                                                round(xyzs[1][2], 5),
                                                                                                                round(sis[1][0], 5),
                                                                                                                round(hbonds[1][0], 5),
                                                                                                                round(charges[1][0], 5),
                                                                                                                round(hphobs[1][0], 5))
        temp_correspond_information = temp_correspond_information + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}\n".format(round(xyzs[2][0], 5),
                                                                                                                round(xyzs[2][1], 5),
                                                                                                                round(xyzs[2][2], 5),
                                                                                                                round(sis[2][0], 5),
                                                                                                                round(hbonds[2][0], 5),
                                                                                                                round(charges[2][0], 5),
                                                                                                                round(hphobs[2][0], 5))
        f_correspond_information.write(temp_correspond_information)

    f_A.close()
    f_graph_indicator.close()
    f_graph_labels.close()
    f_itemID.close()  # 新增：关闭itemID文件
    f_node_attributes.close()
    f_correspond_information.close()

make_database(predict_process_path, 'predict')

100%|██████████| 217/217 [00:02<00:00, 90.70it/s] 


## 检查数据集会不会报错
- 切换环境为dgl

In [13]:
import argparse
from tqdm import tqdm
import dgl
import torch
import torch.nn.functional as F
from dgl.data import tu
from model.encoder import DiffPool

def prepare_data(dataset, shuffle=False, prog_args=None):
    """
    preprocess TU dataset according to DiffPool's paper setting and load dataset into dataloader
    """
    return dgl.dataloading.GraphDataLoader(
        dataset,
        batch_size=prog_args.batch_size,
        shuffle=shuffle,
        num_workers=prog_args.n_worker,
    )


def train(dataset, model, prog_args, id_card_protein):
    """
    training function
    """
    dataloader = dataset

    if prog_args.cuda > 0:
        torch.cuda.set_device(0)
    
    for epoch in range(prog_args.epoch):
        model.train()
        print("\nEPOCH ###### {} ######".format(epoch))
        for batch_idx, (batch_graph, graph_labels, item_ids) in tqdm(enumerate(dataloader), total=len(dataloader)):
            for key, value in batch_graph.ndata.items():
                batch_graph.ndata[key] = value.float()
            graph_labels = graph_labels.long()
            if torch.cuda.is_available():
                batch_graph = batch_graph.to(torch.cuda.current_device())
                graph_labels = graph_labels.cuda()

            model.zero_grad()
            # ypred = model(batch_graph)
            temp = batch_graph
            protein_temp_id_card = ''
            protein_temp_id_card = protein_temp_id_card + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][0][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][6]), 5))
            protein_temp_id_card = protein_temp_id_card + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][1][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][6]), 5))
            protein_temp_id_card = protein_temp_id_card + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][2][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][6]), 5))
            try:
                ypred = model(batch_graph)
            except:
                print('This is the protein name not be forward ==========', id_card_protein[protein_temp_id_card])
                continue
            try:
                ttt = id_card_protein[protein_temp_id_card]
            except:
                print('This is the data not be traced ==========', protein_temp_id_card)
        torch.cuda.empty_cache()
        break
    return 'Trian successfully'


In [10]:
import os
import numpy as np
import dgl
from dgl import DGLGraph
from dgl.data.utils import save_graphs, load_graphs, save_info, load_info
import torch

class customreaddata:
    """
    A custom dataset class to read graph data from specified text files for prediction.
    Assumes the following files exist in raw_dir under a subdirectory named 'name':
    - {name}_A.txt (edge list)
    - {name}_graph_indicator.txt (which graph each node belongs to)
    - {name}_graph_labels.txt (labels for each graph) - This might be dummy for prediction
    - {name}_node_attributes.txt (features for each node)

    Parameters
    ----------
    name : str
        Name of the dataset directory and prefix for files (e.g., 'GTmining').
    raw_dir : str
        Path to the directory containing the dataset folder.
    """

    def __init__(self, name, raw_dir):
        self.name = name
        self.raw_dir = raw_dir
        self.save_dir = os.path.join(raw_dir, name) # Use raw_dir as base, create save path inside
        os.makedirs(self.save_dir, exist_ok=True) # Ensure save directory exists for potential caching

        # Initialize attributes that will be set by process()
        self.graph_lists = []
        self.graph_labels = []
        self.item_ids = []  # 新增：存储每个图的item_id
        self.max_num_node = 0
        self.num_labels = None # May not be relevant for prediction

        # Process the raw data files
        self.process()

    def _file_path(self, category):
        """Constructs the path to a specific data file."""
        return os.path.join(self.raw_dir, f"{self.name}_{category}.txt")

    @staticmethod
    def _idx_from_zero(idx_tensor):
        """Adjusts indices to be 0-based."""
        # Assuming node and graph indices in your files are 1-based.
        # If they are already 0-based, this step is unnecessary or needs adjustment.
        # Check the first few lines of your GTmining_graph_indicator.txt to confirm.
        # If they are 0-based, remove this function or make it a no-op.
        # For now, assuming 1-based as per TUDataset standard.
        min_val = np.min(idx_tensor)
        if min_val == 0:
             # If already 0-based, return as is or handle accordingly
             # This might be the case, adjust logic if needed
             print(f"Warning: Indices in file seem to be 0-based (min={min_val}). Proceeding assuming 0-based.")
             return idx_tensor
        else:
             # Standard 1-based to 0-based conversion
             return idx_tensor - 1 # More standard than subtracting min if known to start at 1

    def process(self):
        """
        Loads data from text files and constructs a list of DGLGraphs.
        """
        print(f"Processing custom dataset: {self.name}")

        # --- 1. Load Edge List ---
        print(f"Loading edges from {self._file_path('A')}")
        # Load edges, assuming 1-based indexing initially
        edge_data_raw = np.genfromtxt(self._file_path("A"), delimiter=",", dtype=int)
        if edge_data_raw.ndim == 1:
            # If only one edge, reshape to (1, 2)
            edge_data_raw = edge_data_raw.reshape(1, -1)
        # Convert to 0-based indices
        edge_data_0_based = self._idx_from_zero(edge_data_raw)
        # DGL expects source and destination arrays
        src_nodes = edge_data_0_based[:, 0]
        dst_nodes = edge_data_0_based[:, 1]

        # --- 2. Load Graph Indicator (which graph each node belongs to) ---
        print(f"Loading graph indicators from {self._file_path('graph_indicator')}")
        node_graph_ids_raw = np.loadtxt(self._file_path("graph_indicator"), dtype=int)
        # Convert graph IDs to 0-based indices
        node_graph_ids = self._idx_from_zero(node_graph_ids_raw)
        num_total_nodes_in_file = len(node_graph_ids_raw)

        # --- 3. Load Graph Labels (might be dummy) ---
        print(f"Loading graph labels from {self._file_path('graph_labels')}")
        try:
            graph_labels_raw = np.loadtxt(self._file_path("graph_labels"), dtype=int)
            # Convert graph labels to 0-based indices if needed, though for classification
            # they often represent class IDs starting from 0 or 1. Adjust if necessary.
            # For prediction, these might just be placeholders.
            self.graph_labels = graph_labels_raw # Keep original values for now, adjust if necessary
            self.num_labels = max(self.graph_labels) + 1 if len(self.graph_labels) > 0 else 0
        except FileNotFoundError:
            print(f"Warning: Graph labels file {self._file_path('graph_labels')} not found. Using dummy labels (e.g., 0).")
            num_graphs_in_file = len(set(node_graph_ids))
            self.graph_labels = np.zeros(num_graphs_in_file, dtype=int) # Dummy labels
            self.num_labels = 1 # Or set to None if not applicable

        # --- 4. Load Item IDs (新增逻辑) ---
        print(f"Loading item IDs from {self._file_path('itemID')}")
        try:
            # 读取itemID文件，支持整数或字符串类型的ID
            self.item_ids = np.loadtxt(self._file_path("itemID"), dtype=str).tolist()
            
            # 验证item_id数量是否与图数量匹配
            num_graphs_in_file = len(set(node_graph_ids))
            if len(self.item_ids) != num_graphs_in_file:
                raise ValueError(
                    f"Number of item IDs ({len(self.item_ids)}) does not match number of graphs ({num_graphs_in_file})."
                )
        except FileNotFoundError:
            print(f"Warning: Item ID file {self._file_path('itemID')} not found. Using graph index as item ID.")
            num_graphs_in_file = len(set(node_graph_ids))
            self.item_ids = [str(i) for i in range(num_graphs_in_file)]  # 使用索引作为默认ID


        # --- 4. Load Node Attributes ---
        print(f"Loading node attributes from {self._file_path('node_attributes')}")
        try:
            node_attributes = np.loadtxt(self._file_path("node_attributes"), delimiter=",")
            if node_attributes.ndim == 1:
                # If features are 1D (one feature per node), reshape to (num_nodes, 1)
                node_attributes = np.expand_dims(node_attributes, axis=1)
            print(f"Loaded node attributes with shape: {node_attributes.shape}")
            if node_attributes.shape[0] != num_total_nodes_in_file:
                 raise ValueError(f"Number of rows in node_attributes ({node_attributes.shape[0]}) does not match number of nodes indicated by graph_indicator ({num_total_nodes_in_file}).")
        except FileNotFoundError:
            print(f"Warning: Node attributes file {self._file_path('node_attributes')} not found. Graphs will have no node features (ndata['feat'] will not be set initially).")
            node_attributes = None


        # --- 5. Create a Base Graph with All Nodes and Edges ---
        # This graph contains all nodes from all graphs, connected by the provided edges.
        num_nodes_in_base_graph = int(np.max(src_nodes)) + 1 if len(src_nodes) > 0 else 0
        # Ensure num_nodes includes any isolated nodes that might only appear in graph_indicator
        num_nodes_in_base_graph = max(num_nodes_in_base_graph, num_total_nodes_in_file)

        if num_nodes_in_base_graph == 0:
            print("Warning: No nodes or edges found in the data files.")
            self.graph_lists = []
            return

        base_graph = dgl.graph(([], []), num_nodes=num_nodes_in_base_graph)
        base_graph.add_edges(src_nodes, dst_nodes)

        # Assign node attributes to the base graph if available
        if node_attributes is not None:
            base_graph.ndata['feat'] = torch.tensor(node_attributes, dtype=torch.float32)

        # --- 6. Split the Base Graph into Individual Graphs ---
        self.graph_lists = []
        self.max_num_node = 0

        num_expected_graphs = len(set(node_graph_ids))
        print(f"Found {num_expected_graphs} graphs based on graph_indicator.")

        for graph_id in range(num_expected_graphs):
            # Find the nodes belonging to the current graph (graph_id)
            node_mask = (node_graph_ids == graph_id)
            node_indices_for_graph = np.where(node_mask)[0] # Get 0-based indices of nodes in this graph

            if len(node_indices_for_graph) == 0:
                print(f"Warning: Graph ID {graph_id} has no nodes according to graph_indicator.")
                # Create an empty graph for this ID
                g_sub = dgl.graph(([], []), num_nodes=0)
                # Add a dummy feature tensor if original had features, though shape might be tricky for 0 nodes
                # Often, empty graphs might need special handling downstream.
                # For now, just create the empty graph.
            else:
                # Extract the subgraph corresponding to these nodes
                g_sub = base_graph.subgraph(node_indices_for_graph)

                # The subgraph's nodes have new IDs (0, 1, ...). The original features are preserved based on the subgraph operation.
                # If node_attributes was loaded, 'feat' is already in g_sub.ndata.
                # Check if 'feat' exists, otherwise features were not available.
                if 'feat' not in g_sub.ndata:
                     print(f"  Graph {graph_id}: No node features available.")


            self.graph_lists.append(g_sub)

            if g_sub.num_nodes() > self.max_num_node:
                self.max_num_node = g_sub.num_nodes()

        print(f"Successfully processed {len(self.graph_lists)} graphs.")
        print(f"Max number of nodes in a single graph: {self.max_num_node}")


    def __getitem__(self, idx):
        """
        Gets the graph and its label at the given index.

        Parameters
        ---------
        idx : int
            The sample index.

        Returns
        -------
        dgl.DGLGraph
            The graph object, potentially with node features in `ndata['feat']`.
        torch.Tensor
            The label tensor for the graph (could be dummy for prediction).
        """
        if idx < 0 or idx >= len(self):
             raise IndexError(f"Index {idx} is out of range for dataset with {len(self)} items.")
        g = self.graph_lists[idx]
        label = torch.tensor(self.graph_labels[idx], dtype=torch.int64) if self.graph_labels is not None else torch.tensor(0, dtype=torch.int64) # Return a dummy label if not set
        item_id = self.item_ids[idx]  # 新增：获取对应索引的item_id

        return g, label, item_id

    def __len__(self):
        """
        Returns the number of graphs in the dataset.
        """
        return len(self.graph_lists)

    @property
    def num_classes(self):
        """Returns the number of classes (uses num_labels)."""
        return int(self.num_labels) if self.num_labels is not None else 0 # Return 0 if not set




dataset_predict = customreaddata(
    name="GTmining",
    raw_dir='./predict_data/dl_data/predict/' # Adjust path as necessary
)

print(f"Dataset length: {len(dataset_predict)}")
if len(dataset_predict) > 0:
    g, label, item_id = dataset_predict[0]
    print(f"First graph: {g}")
    print(f"First graph label: {label}")
    print(f"First graph item ID: {item_id}")
    print(f"First graph node features shape: {g.ndata.get('feat', None).shape if 'feat' in g.ndata else 'N/A'}")


Processing custom dataset: GTmining
Loading edges from ./predict_data/dl_data/predict/GTmining_A.txt
Loading graph indicators from ./predict_data/dl_data/predict/GTmining_graph_indicator.txt
Loading graph labels from ./predict_data/dl_data/predict/GTmining_graph_labels.txt
Loading item IDs from ./predict_data/dl_data/predict/GTmining_itemID.txt
Loading node attributes from ./predict_data/dl_data/predict/GTmining_node_attributes.txt
Loaded node attributes with shape: (91905, 7)
Found 217 graphs based on graph_indicator.
Successfully processed 217 graphs.
Max number of nodes in a single graph: 512
Dataset length: 217
First graph: Graph(num_nodes=468, num_edges=1287,
      ndata_schemes={'feat': Scheme(shape=(7,), dtype=torch.float32), '_ID': Scheme(shape=(), dtype=torch.int64)}
      edata_schemes={'_ID': Scheme(shape=(), dtype=torch.int64)})
First graph label: 0
First graph item ID: ACD36995.1_d
First graph node features shape: torch.Size([468, 7])


In [11]:
sample_redies_udp = 6
sample_redies_sugar = 6
family_fold_type = 'GTB'
fold_num = 1


global_train_time_per_epoch = []

print("{:=^100}".format('prog_args'))
global_train_time_per_epoch = []
prog_args = argparse.Namespace(dataset=f'GTmining_6_6_{family_fold_type}_fold{fold_num}', pool_ratio=0.10, num_pool=1, cuda=1, lr=1e-4, clip=1.0,
                               batch_size=1, epoch=1, train_ratio=0.7, test_ratio=0.1, n_worker=2, gc_per_block=3,
                               dropout=0.20, method="diffpool", bn=True, bias=True, save_dir="./model_param_alldata",
                               load_epoch=-1, data_mode="default", linkpred=False, hidden_dim=64, embedding_dim=64)
print(prog_args)

print("{:=^100}".format('加载数据'))
dataset_train = tu.LegacyTUDataset(name="GTmining",
                                    raw_dir=f'../data/dl_data/{family_fold_type}_alldata/fold{fold_num}/train/')
dataset_validation = tu.LegacyTUDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}_alldata/fold{fold_num}/validation/')
dataset_test = tu.LegacyTUDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}_alldata/fold{fold_num}/test/')
train_dataloader = prepare_data(dataset_train, shuffle=True, prog_args=prog_args)
validation_dataloader = prepare_data(dataset_validation, shuffle=False, prog_args=prog_args)
test_dataloader = prepare_data(dataset_test, shuffle=False, prog_args=prog_args)

dataset_predict = customreaddata(name="GTmining",
                                   raw_dir=f'./predict_data/dl_data/predict/')
predict_dataloader = prepare_data(dataset_predict, shuffle=False, prog_args=prog_args)

input_dim_train, label_dim_train, max_num_node_train = dataset_train.statistics()
input_dim_validation, label_dim_validation, max_num_node_validation = dataset_validation.statistics()
input_dim_test, label_dim_test, max_num_node_test = dataset_test.statistics()
max_num_node = max([max_num_node_train, max_num_node_validation, max_num_node_test])
input_dim = input_dim_train
label_dim = label_dim_train
print("++++++++++ STATISTICS ABOUT THE DATASET ++++++++++")
print("dataset feature dimension is", input_dim_train)
print("dataset label dimension is", label_dim_train)
print("the max num node is", max_num_node)
print("number of graphs is", len(dataset_train) + len(dataset_validation)+ len(dataset_test))

hidden_dim = prog_args.hidden_dim  # used to be 64
embedding_dim = prog_args.embedding_dim

assign_dim = int(max_num_node * prog_args.pool_ratio)
print("++++++++++MODEL STATISTICS++++++++")
print("model hidden dim is", hidden_dim)
print("model embedding dim for graph instance embedding", embedding_dim)
print("initial batched pool graph dim is", assign_dim)
activation = F.relu

# initialize model
model = DiffPool(
    input_dim,
    hidden_dim,
    embedding_dim,
    label_dim,
    activation,
    prog_args.gc_per_block,
    prog_args.dropout,
    prog_args.num_pool,
    prog_args.linkpred,
    prog_args.batch_size,
    "meanpool",
    assign_dim,
    prog_args.pool_ratio,
    []
)

print("model init finished")
print("MODEL:::::::", prog_args.method)
if prog_args.cuda:
    model = model.cuda()

=============================================prog_args==============================================
Namespace(dataset='GTmining_6_6_GTB_fold1', pool_ratio=0.1, num_pool=1, cuda=1, lr=0.0001, clip=1.0, batch_size=1, epoch=1, train_ratio=0.7, test_ratio=0.1, n_worker=2, gc_per_block=3, dropout=0.2, method='diffpool', bn=True, bias=True, save_dir='./model_param_alldata', load_epoch=-1, data_mode='default', linkpred=False, hidden_dim=64, embedding_dim=64)
================================================加载数据================================================
Processing custom dataset: GTmining
Loading edges from ./predict_data/dl_data/predict/GTmining_A.txt
Loading graph indicators from ./predict_data/dl_data/predict/GTmining_graph_indicator.txt
Loading graph labels from ./predict_data/dl_data/predict/GTmining_graph_labels.txt
Loading item IDs from ./predict_data/dl_data/predict/GTmining_itemID.txt
Loading node attributes from ./predict_data/dl_data/predict/GTmining_node_attributes.txt
Loaded

In [14]:
# train_dataloader  validation_dataloader  test_dataloader  nova_dataloader  predict_dataloader
# train  validation  test  nova  predict

id_card_protein = {}
with open(f'./predict_data/dl_data/predict/Predict_correspond_information.txt', 'r')as f:
    for dd in f.readlines():
        dd = dd.split('\n')[0].split('===')
        id_card_protein[dd[1]] = dd[0]

logger = train(
    predict_dataloader, model, prog_args, id_card_protein
)

print("Ending!!!")


EPOCH ###### 0 ######


100%|██████████| 217/217 [00:08<00:00, 26.27it/s]

Ending!!!


# 模型预测

In [15]:
fold_num = 1
family_fold_type = 'GTB'

## 函数定义

In [16]:
import argparse
import textwrap
import os
import time
import dgl
import torch
import torch.nn as nn
import torch.nn.functional as F
from dgl.data import tu
from model.encoder import DiffPool
from livelossplot import PlotLosses
from sklearn.metrics import f1_score
import shutil
import pandas as pd

def prepare_data(dataset, shuffle=False, prog_args=None, custom_batch_size=None):
    """
    preprocess TU dataset according to DiffPool's paper setting and load dataset into dataloader
    """
    if custom_batch_size is None:
        return dgl.dataloading.GraphDataLoader(
            dataset,
            batch_size=prog_args.batch_size,
            shuffle=shuffle,
            num_workers=prog_args.n_worker,
        )
    else:
        return dgl.dataloading.GraphDataLoader(
            dataset,
            batch_size=custom_batch_size,
            shuffle=shuffle,
            num_workers=prog_args.n_worker,
        )


print("{:=^100}".format(f'fold num is : {fold_num}, family type is : {family_fold_type}'))

print("{:=^100}".format('prog_args'))
prog_args = argparse.Namespace(dataset=f'GTmining_6_6_{family_fold_type}_fold{fold_num}', pool_ratio=0.10, num_pool=1, cuda=1, lr=1.0, clip=float("inf"),
                               batch_size=128, epoch=500, n_worker=10, gc_per_block=3, aggregator_type="meanpool",
                               dropout=0.00, method="diffpool", bn=True, bias=True, save_dir=f"./model_param",
                               load_epoch=-1, data_mode="default", linkpred=False, hidden_dim=64, embedding_dim=64, family_fold_type=family_fold_type)
print( textwrap.fill(str(prog_args), width=100))

print("{:=^100}".format('加载数据'))
dataset_train = tu.LegacyTUDataset(name="GTmining",
                                    raw_dir=f'../data/dl_data/{family_fold_type}_alldata/fold{fold_num}/train/')
dataset_validation = tu.LegacyTUDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}_alldata/fold{fold_num}/validation/')
dataset_test = tu.LegacyTUDataset(name="GTmining",
                                   raw_dir=f'../data/dl_data/{family_fold_type}_alldata/fold{fold_num}/test/')
train_dataloader = prepare_data(dataset_train, shuffle=True, prog_args=prog_args)
validation_dataloader = prepare_data(dataset_validation, shuffle=False, prog_args=prog_args)
test_dataloader = prepare_data(dataset_test, shuffle=False, prog_args=prog_args)

dataset_predict = customreaddata(name="GTmining",
                                   raw_dir=f'./predict_data/dl_data/predict/')
predict_dataloader = prepare_data(dataset_predict, shuffle=False, prog_args=prog_args, custom_batch_size=1)


input_dim_train, label_dim_train, max_num_node_train = dataset_train.statistics()
input_dim_validation, label_dim_validation, max_num_node_validation = dataset_validation.statistics()
input_dim_test, label_dim_test, max_num_node_test = dataset_test.statistics()
max_num_node = max([max_num_node_train, max_num_node_validation, max_num_node_test])
input_dim = input_dim_train
label_dim = label_dim_train
print("++++++++++ STATISTICS ABOUT THE DATASET ++++++++++")
print("dataset feature dimension is", input_dim_train)
print("dataset label dimension is", label_dim_train)
print("the max num node is", max_num_node)
print("number of graphs is", len(dataset_train) + len(dataset_validation)+ len(dataset_test))

hidden_dim = prog_args.hidden_dim  # used to be 64
embedding_dim = prog_args.embedding_dim

assign_dim = int(max_num_node * prog_args.pool_ratio)
print("++++++++++MODEL STATISTICS++++++++")
print("model hidden dim is", hidden_dim)
print("model embedding dim for graph instance embedding", embedding_dim)
print("initial batched pool graph dim is", assign_dim)
activation = F.relu

if family_fold_type == 'GTA':
    graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                        'UDP-Gal': 3, 'UDP-GalNAc': 4,
                        'UDP-Xyl': 5, 'GDP-Man': 6,
                        'dTDP-Rha': 7, 'Other': 8}
elif family_fold_type == 'GTB':
    graph_label_dict = {'UDP-Glc': 0, 'UDP-GlcNAc': 1, 'UDP-GlcA': 2,
                        'UDP-Gal': 3, 'UDP-GalNAc': 4,
                        'UDP-Xyl': 5, 'GDP-Man': 6, 'GDP-Fuc': 7,
                        'dTDP-Rha': 8, 'Other': 9}
else:
    raise ValueError(f"Invalid family_fold_type: '{prog_args.family_fold_type}'. Valid options are 'GTA' and 'GTB'.")
df_cluster = pd.read_excel(f'../data/cluster/{family_fold_type}/dataseat_split_{fold_num}.xlsx')
df_cluster = df_cluster.loc[df_cluster['Dataset']=='train']
df_cluster.reset_index(drop=True, inplace=True)
custom_loss_weight = []
total_sample = df_cluster.shape[0]
for x in graph_label_dict.keys():
    df_x = df_cluster.loc[df_cluster['Activate']==x]
    df_x.reset_index(drop=True, inplace=True)
    x_sample = df_x.shape[0]
    custom_loss_weight.append(total_sample/x_sample)

assert len(custom_loss_weight) == label_dim_train, 'Wrong custom loss weight, please check what happen.'

# initialize model
model = DiffPool(
    input_dim,
    hidden_dim,
    embedding_dim,
    label_dim,
    activation,
    prog_args.gc_per_block,
    prog_args.dropout,
    prog_args.num_pool,
    prog_args.linkpred,
    prog_args.batch_size,
    prog_args.aggregator_type,
    assign_dim,
    prog_args.pool_ratio,
    custom_loss_weight
)
print("model init finished")
print("MODEL:::::::", prog_args.method)
if prog_args.cuda:
    model = model.cuda()




===============================fold num is : 1, family type is : GTB================================
=============================================prog_args==============================================
Namespace(dataset='GTmining_6_6_GTB_fold1', pool_ratio=0.1, num_pool=1, cuda=1, lr=1.0, clip=inf,
batch_size=128, epoch=500, n_worker=10, gc_per_block=3, aggregator_type='meanpool', dropout=0.0,
method='diffpool', bn=True, bias=True, save_dir='./model_param', load_epoch=-1, data_mode='default',
linkpred=False, hidden_dim=64, embedding_dim=64, family_fold_type='GTB')
================================================加载数据================================================
Processing custom dataset: GTmining
Loading edges from ./predict_data/dl_data/predict/GTmining_A.txt
Loading graph indicators from ./predict_data/dl_data/predict/GTmining_graph_indicator.txt
Loading graph labels from ./predict_data/dl_data/predict/GTmining_graph_labels.txt
Loading item IDs from ./predict_data/dl_data/predict/G

In [18]:
# 先过一遍train_dataloader，让模型中的一些参数先初始化一下
model.train()
for batch_idx, (batch_graph, graph_labels) in enumerate(train_dataloader):
    for key, value in batch_graph.ndata.items():
        batch_graph.ndata[key] = value.float()
    graph_labels = graph_labels.long()
    if torch.cuda.is_available():
        batch_graph = batch_graph.to(torch.cuda.current_device())
        graph_labels = graph_labels.cuda()
    ypred = model(batch_graph)
    loss = model.loss(ypred, graph_labels)
    loss.backward()
    nn.utils.clip_grad_norm_(model.parameters(), max_norm=prog_args.clip)
    model.zero_grad()



## 开始预测

In [19]:
import pandas as pd

GTA_best_epoch = {}
for x in range(1, 11):
    df_path = os.path.join(prog_args.save_dir, f'GTmining_6_6_GTA_fold{x}', 'validation_log.csv')
    df = pd.read_csv(df_path)
    best_epoch = df.loc[df['validation_f1_score'].idxmax(), 'epoch']
    GTA_best_epoch[x] = int(best_epoch)

GTB_best_epoch = {}
for x in range(1, 11):
    df_path = os.path.join(prog_args.save_dir, f'GTmining_6_6_GTB_fold{x}', 'validation_log.csv')
    df = pd.read_csv(df_path)
    best_epoch = df.loc[df['validation_f1_score'].idxmax(), 'epoch']
    GTB_best_epoch[x] = int(best_epoch)
    xx = df['validation_f1_score'][best_epoch]
    xxx = df['test_f1_score'][best_epoch]
    xxxx = df['nova_f1_score'][best_epoch]
    print(f'Best epoch for GTB fold {x} is {best_epoch}, validation f1 score is {xx}, test f1 score is {xxx}, nova f1 score is {xxxx}.')

id_card_protein = {}
with open(f'./predict_data/dl_data/predict/Predict_correspond_information.txt', 'r')as f:
    for dd in f.readlines():
        dd = dd.split('\n')[0].split('===')
        id_card_protein[dd[1]] = dd[0]


GTA_best_epoch = {
        1: 367, 2: 764, 3: 631, 4: 570,
        5: 451, 6: 473, 7: 704, 8: 292,
        9: 364, 10: 353
    }
GTB_best_epoch = {
        1: 206, 2: 245, 3: 152, 4: 141,
        5: 225, 6: 137, 7: 119, 8: 216,
        9: 205, 10: 143
    }


print(GTA_best_epoch)
print(GTB_best_epoch)



Best epoch for GTB fold 1 is 316, validation f1 score is 0.8234489104235339, test f1 score is 0.728960034172425, nova f1 score is 0.1160439761979334.
Best epoch for GTB fold 2 is 447, validation f1 score is 0.7628767893837057, test f1 score is 0.7657175823345544, nova f1 score is 0.0667500577007975.
Best epoch for GTB fold 3 is 419, validation f1 score is 0.785752892022684, test f1 score is 0.8364906139159123, nova f1 score is 0.0942394216163972.
Best epoch for GTB fold 4 is 491, validation f1 score is 0.8726463205008012, test f1 score is 0.659521201330241, nova f1 score is 0.0911931159152793.
Best epoch for GTB fold 5 is 499, validation f1 score is 0.8371655750671397, test f1 score is 0.7027207428760833, nova f1 score is 0.1712017311954953.
Best epoch for GTB fold 6 is 338, validation f1 score is 0.7856974241406034, test f1 score is 0.7925952564452078, nova f1 score is 0.1625316793881905.
Best epoch for GTB fold 7 is 479, validation f1 score is 0.7384276297221521, test f1 score is 0.7

In [28]:
from collections import defaultdict, Counter

if family_fold_type == 'GTA':
    best_epoch_dict = GTA_best_epoch
elif family_fold_type == 'GTB':
    best_epoch_dict = GTB_best_epoch
else:
    raise ValueError('SSSSSB')

predict_result = defaultdict(list)
predict_result_freq = defaultdict(list)

for fold_num_predict in range(1, len(list(best_epoch_dict.keys()))+1):
    result_index = 1
    epoch_predict = best_epoch_dict[fold_num_predict]

    begin_time = time.time()
    print("\nEPOCH ###### Fold {} Epoch {} ######".format(fold_num_predict, epoch_predict))
    if epoch_predict is not None and prog_args.save_dir is not None:
        model.load_state_dict(
            torch.load(
                prog_args.save_dir
                + "/"
                + prog_args.dataset
                + "/model.iter-"
                + "{:04d}".format(epoch_predict), weights_only=True
            )
        )


    model.eval()
    with torch.no_grad():
        val_pred_indi = torch.tensor([], device='cuda')
        val_label_indi = torch.tensor([], device='cuda')
        for batch_idx, (batch_graph, graph_labels, item_id) in enumerate(predict_dataloader):
            for key, value in batch_graph.ndata.items():
                batch_graph.ndata[key] = value.float()
            graph_labels = graph_labels.long()
            if torch.cuda.is_available():
                batch_graph = batch_graph.to(torch.cuda.current_device())
                graph_labels = graph_labels.cuda()

            # 拿标签
            temp = batch_graph
            protein_temp_id_card = ''
            protein_temp_id_card = protein_temp_id_card + "{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][0][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][0][6]), 5))
            protein_temp_id_card = protein_temp_id_card + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][1][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][1][6]), 5))
            protein_temp_id_card = protein_temp_id_card + "###{:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}, {:>.2f}".format(round(float(temp.ndata['feat'][2][0]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][1]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][2]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][3]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][4]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][5]), 5),
                                                                                                                                    round(float(temp.ndata['feat'][2][6]), 5))
            # protein_id = id_card_protein[protein_temp_id_card]
            print(f"Protein ID: {item_id[0]}, Protein temp id card: {id_card_protein[protein_temp_id_card]}")
            protein_id = item_id[0]


            ypred = model(batch_graph)
            indi = torch.argmax(ypred, dim=1)
            predict_result[protein_id+f'_{result_index}'].append(int(indi.cpu()))
            predict_result_freq[protein_id+f'_{result_index}'].append(F.softmax(ypred, dim=1).cpu().tolist()[0])
            result_index += 1
            # print(f"Protein name is : {protein_id}, and its activate is {int(indi.cpu())}.")
            
    elapsed_time = time.time() - begin_time
    print("epoch {:.4f} with epoch time {:.4f} s".format(epoch_predict, elapsed_time))

    break


EPOCH ###### Fold 1 Epoch 206 ######
Protein ID: ACD36995.1_d, Protein temp id card: ACD36995.1_d
Protein ID: BAQ01014.1_d, Protein temp id card: ACD75811.1_d
Protein ID: AIG62608.1_d, Protein temp id card: AIG62608.1_d
Protein ID: BAQ01353.1_d, Protein temp id card: AAV85953.1_d
Protein ID: AIG62606.1_d, Protein temp id card: AIG62606.1_d
Protein ID: AIG62628.1_d, Protein temp id card: AIG62628.1_d
Protein ID: AAV74375.1_d, Protein temp id card: AAV74375.1_d
Protein ID: ADR74238.1_d, Protein temp id card: ADR74238.1_d
Protein ID: AIG62749.1_d, Protein temp id card: AIG62749.1_d
Protein ID: AAV85953.1_d, Protein temp id card: AAV85953.1_d
Protein ID: AIG62836.1_d, Protein temp id card: AIG62836.1_d
Protein ID: ABM53650.1_d, Protein temp id card: ABM53662.1_d
Protein ID: BAQ00691.1_d, Protein temp id card: BAQ00691.1_d
Protein ID: ACA24823.1_d, Protein temp id card: ACA24823.1_d
Protein ID: AIG56947.1_d, Protein temp id card: AIG56947.1_d
Protein ID: AIG62426.1_d, Protein temp id card:

In [21]:
temp_predict_result_freq = defaultdict(list)

for key in predict_result_freq.keys():
    freq_list = np.array(predict_result_freq[key])
    freq_list = freq_list.reshape(-1, len(graph_label_dict)).round(2)
    temp_predict_result_freq[key] = freq_list.tolist()

predict_result_freq = temp_predict_result_freq




In [26]:
item_id

('ACD36995.1_d',)

In [15]:
# 存储结果：key为原键，value为(元素, 出现次数)的列表
result = {}

for key, values in predict_result.items():
    # 统计每个元素的出现次数
    count = Counter(values)
    
    if not count:  # 处理空列表情况（示例中没有空列表，仅作兼容）
        result[key] = []
        continue
    
    # 找到最大出现次数
    max_freq = max(count.values())
    
    # 筛选出所有出现次数等于最大次数的元素
    most_common = [(num, freq) for num, freq in count.items() if freq == max_freq]
    
    result[key] = most_common

# 打印结果
for key, items in result.items():
    print(f"{key}:")
    print(predict_result[key])
    print("预测概率分布")
    for i in range(len(predict_result_freq[key])):
        print(predict_result_freq[key][i])
    for num, freq in items:
        print(f"  元素 {num}，出现次数 {freq}")
    print()  # 空行分隔

ACD36995.1_d_1:
[6, 9, 9, 6, 6, 9, 6, 6, 9, 1]
预测概率分布
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.99]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.94, 0.0, 0.0, 0.06]
[0.0, 0.35, 0.0, 0.0, 0.0, 0.0, 0.65, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.41, 0.0, 0.0, 0.0, 0.0, 0.46, 0.0, 0.0, 0.12]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.01, 0.0, 0.0, 0.99]
[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  元素 6，出现次数 5

ACD75811.1_d_2:
[9, 9, 9, 9, 9, 9, 9, 9, 9, 9]
预测概率分布
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]


In [16]:
import pandas as pd

# 存储结果：key为原键，value为(元素, 出现次数)的列表
result = {}

for key, values in predict_result.items():
    # 统计每个元素的出现次数
    count = Counter(values)
    
    if not count:  # 处理空列表情况（示例中没有空列表，仅作兼容）
        result[key] = []
        continue
    
    # 找到最大出现次数
    max_freq = max(count.values())
    
    # 筛选出所有出现次数等于最大次数的元素
    most_common = [(num, freq) for num, freq in count.items() if freq == max_freq]
    
    result[key] = most_common

# 打印结果
for key, items in result.items():
    print(f"{key}:")
    print(predict_result[key])
    print("预测概率分布")
    for i in range(len(predict_result_freq[key])):
        print(predict_result_freq[key][i])
    for num, freq in items:
        print(f"  元素 {num}，出现次数 {freq}")
    print()  # 空行分隔


# 将结果存储为Excel表格

# 反转graph_label_dict
label_to_name = {v: k for k, v in graph_label_dict.items()}

# 准备数据
data = []
for key, items in result.items():
    predictions = predict_result[key]
    for num, freq in items:
        label_name = label_to_name.get(num, "Unknown")
        data.append([key, predictions, f"元素 {label_name}，出现次数 {freq}"])

# 创建DataFrame
df_result = pd.DataFrame(data, columns=["Key", "Predictions", "Most Common Element"])

# 保存为Excel文件
df_result.to_excel("./predict_data/prediction_results.xlsx", index=False)
print("结果已保存为 ./predict_data/prediction_results.xlsx")

ACD36995.1_d_1:
[6, 9, 9, 6, 6, 9, 6, 6, 9, 1]
预测概率分布
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.99]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.94, 0.0, 0.0, 0.06]
[0.0, 0.35, 0.0, 0.0, 0.0, 0.0, 0.65, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.41, 0.0, 0.0, 0.0, 0.0, 0.46, 0.0, 0.0, 0.12]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.01, 0.0, 0.0, 0.99]
[0.0, 1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]
  元素 6，出现次数 5

ACD75811.1_d_2:
[9, 9, 9, 9, 9, 9, 9, 9, 9, 9]
预测概率分布
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 1.0]
